# MDON - Shinnecock Inlet

In [ ]:
import os
import sys
import tensorflow as tf
tf.keras.backend.set_floatx('float64') 


import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, ScalarFormatter, FormatStrFormatter
from matplotlib import ticker as mtick
import cmocean

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error as mae
import joblib

from IPython.display import display
import matplotlib.ticker as ticker
from matplotlib import rcParams
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import netCDF4 as nc
from tqdm import tqdm

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.patches as mpatches

plt.rc('font', family='serif')
plt.rcParams.update({'font.size': 20,
                     'lines.linewidth': 2,
                     'lines.markersize':10,
                     'axes.labelsize': 16, # fontsize for x and y labels (was 10)
                     'axes.titlesize': 20,
                     'xtick.labelsize': 16,
                     'ytick.labelsize': 16,
                     'legend.fontsize': 16,
                     'axes.linewidth': 2})

import itertools
colors = itertools.cycle(['r','g','b','m','y','c'])
markers = itertools.cycle(['p','d','o','^','s','x',]) #'D','H','v','*'])

from pathlib import Path
try:
    work_dir.exists()
except NameError:
    curr_dir = Path().resolve()
    work_dir = curr_dir.parent  

scripts_dir = work_dir / "src"
sys.path.append(str(scripts_dir.absolute()))

# mdon_model_dir = work_dir / "saved_models"
# scalers_dir = work_dir / "scalers"
mdon_model_dir = work_dir / "saved_models_mdon"
scalers_dir = work_dir / "scalers_mdon"

#Download and copy your data to this folder
ROOT_DIR = work_dir.parents[1]
SHINNECOCK_DATA_DIR = (
    ROOT_DIR
    / 'data'
    / 'PRJ-5716'
    / 'Simulation--2d-adcirc-simulation-of-tidal-flow-in-shinnecock-inlet-ny-parameterized-by-bottom-friction-coefficient'
    / 'data'
    / 'Model--adcirc-model'
    / 'data'
)
data_dir = SHINNECOCK_DATA_DIR


import mdon 
import data_loader as dl 
import processing_utils as pu

from importlib import reload as reload

%matplotlib inline

## Specify training configuraiton for scaling purposes and testing cases for visualization

In [ ]:
# Training data for scaling
train_days = [15., 45.]
param_list_train = [0.003, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05]
scaling = True

# Test data
test_days = [5., 45.]
param_list_test = [0.0025, 0.00275, 0.003, 0.005, 0.0075, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05, 0.075, 0.1]
# unseen_param_list_test = [0.0025, 0.015, 0.1]
unseen_param_loc = [0,6,-1]

window_size = 5

## Load Models

In [ ]:
# Load H Models
mdon_model_path_h = str(mdon_model_dir) + '/MDON_h'
mdon_model_h = tf.keras.models.load_model(mdon_model_path_h, compile=False)

# Load U Models
mdon_model_path_u = str(mdon_model_dir) + '/MDON_u'
mdon_model_u = tf.keras.models.load_model(mdon_model_path_u, compile=False)

# Load V Models
mdon_model_path_v = str(mdon_model_dir) + '/MDON_v'
mdon_model_v = tf.keras.models.load_model(mdon_model_path_v, compile=False)

### Print Model Summary

In [ ]:
# Print MDONNet model summary
mdon_model_h.summary()
mdon_model_u.summary()
mdon_model_v.summary()

In [ ]:
# Load mesh
mesh = dl.load_mesh(data_dir/"cf0025")

# Extract fields
field = mesh['bathy']
triangles = mesh['triangles'] - 1  # if your indexing in 'triangles' starts at 1, convert to 0-based
lon = mesh['nodes'][:, 0]
lat = mesh['nodes'][:, 1]

# Sensors (node indices)
sensors = [2775, 2624, 1115]
s1_color = cmocean.cm.phase(0.6)
s2_color = cmocean.cm.phase(1)
s3_color = cmocean.cm.phase(0.8)

# Compute valmin and valmax from valid (unmasked) data
valmax = field[~field.mask].max()
valmin = field[~field.mask].min()

# ---- Single figure with two subplots ----
fig = plt.figure(figsize=(30,20))

# LEFT: full domain
ax = plt.subplot(1, 2, 1, projection=ccrs.PlateCarree())

# Esri World Imagery (satellite) via WMTS
wmts_url = "https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer/WMTS/1.0.0/WMTSCapabilities.xml"
ax.add_wmts(wmts_url, layer_name="World_Imagery", zorder=0)

# Plot the bathymetry
mesh_plot_left = ax.tripcolor(
    lon, 
    lat, 
    triangles, 
    field, 
    cmap='cmo.deep', 
    edgecolor='k',
    vmin=valmin, 
    vmax=valmax,
    transform=ccrs.PlateCarree()
)

# Desired map extent (lon_min, lon_max, lat_min, lat_max)
extent = [-73.0, -71.95, 40.3, 41.05]
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Customize tick labels
ax.set_xticks([-73, -72.5, -72], crs=ccrs.PlateCarree())
ax.set_yticks([40.4, 40.6, 40.8, 41.0], crs=ccrs.PlateCarree())
lon_formatter = LongitudeFormatter(degree_symbol='°')
lat_formatter = LatitudeFormatter(degree_symbol='°')
ax.xaxis.set_major_formatter(lon_formatter)
ax.yaxis.set_major_formatter(lat_formatter)

# Add sensor markers (noticeable, non-circular)
ax.scatter(lon[sensors[0]], lat[sensors[0]], s=150, marker='o', color=s1_color,   label='Sensor 1', transform=ccrs.PlateCarree(), zorder=3)
ax.scatter(lon[sensors[1]], lat[sensors[1]], s=150, marker='o', color=s2_color, label='Sensor 2', transform=ccrs.PlateCarree(), zorder=3)
ax.scatter(lon[sensors[2]], lat[sensors[2]], s=150, marker='o', color=s3_color,  label='Sensor 3', transform=ccrs.PlateCarree(), zorder=3)
ax.legend(loc='upper left')

# RIGHT: zoomed-in
ax2 = plt.subplot(1, 2, 2, projection=ccrs.PlateCarree())

# Esri World Imagery (satellite) via WMTS
ax2.add_wmts(wmts_url, layer_name="World_Imagery", zorder=0)

# Plot the bathymetry (same vmin/vmax for shared colorbar)
mesh_plot_right = ax2.tripcolor(
    lon, 
    lat, 
    triangles, 
    field, 
    cmap='cmo.deep', 
    edgecolor='k',
    vmin=valmin, 
    vmax=valmax,
    transform=ccrs.PlateCarree()
)

# Desired map extent (lon_min, lon_max, lat_min, lat_max)
extent_zoom = [-72.55, -72.4, 40.8, 40.9]
ax2.set_extent(extent_zoom, crs=ccrs.PlateCarree())

# Customize tick labels
ax2.set_xticks([-72.54, -72.48, -72.4], crs=ccrs.PlateCarree())
ax2.set_yticks([40.8, 40.85, 40.9], crs=ccrs.PlateCarree())
ax2.xaxis.set_major_formatter(lon_formatter)
ax2.yaxis.set_major_formatter(lat_formatter)

# Add sensor markers to zoom panel (same markers/colors)
ax2.scatter(lon[sensors[0]], lat[sensors[0]], s=150, marker='o', color=s1_color,   label='Sensor 1', transform=ccrs.PlateCarree(), zorder=3)
ax2.scatter(lon[sensors[1]], lat[sensors[1]], s=150, marker='o', color=s2_color, label='Sensor 2', transform=ccrs.PlateCarree(), zorder=3)
ax2.scatter(lon[sensors[2]], lat[sensors[2]], s=150, marker='o', color=s3_color,  label='Sensor 3', transform=ccrs.PlateCarree(), zorder=3)

# Single colorbar on the right (attach to both axes)
cbar = fig.colorbar(mesh_plot_right, ax=[ax, ax2], shrink=0.4, pad=0.04)
cbar.set_label('Depth (m)')

plt.show()

## Load Scalers

In [ ]:
tt_scaler_h = joblib.load(str(scalers_dir)+'/tt_scaler_h.save') 
tx_scaler_h = joblib.load(str(scalers_dir)+'/tx_scaler_h.save') 
ty_scaler_h = joblib.load(str(scalers_dir)+'/ty_scaler_h.save') 

b_scaler_h = joblib.load(str(scalers_dir)+'/b_scaler_h.save') 
target_scaler_h = joblib.load(str(scalers_dir)+'/target_scaler_h.save') 

tt_scaler_u = joblib.load(str(scalers_dir)+'/tt_scaler_u.save') 
tx_scaler_u = joblib.load(str(scalers_dir)+'/tx_scaler_u.save') 
ty_scaler_u = joblib.load(str(scalers_dir)+'/ty_scaler_u.save') 

b_scaler_u = joblib.load(str(scalers_dir)+'/b_scaler_u.save') 
target_scaler_u = joblib.load(str(scalers_dir)+'/target_scaler_u.save') 

tt_scaler_v = joblib.load(str(scalers_dir)+'/tt_scaler_v.save') 
tx_scaler_v = joblib.load(str(scalers_dir)+'/tx_scaler_v.save') 
ty_scaler_v = joblib.load(str(scalers_dir)+'/ty_scaler_v.save') 

b_scaler_v = joblib.load(str(scalers_dir)+'/b_scaler_v.save') 
target_scaler_v = joblib.load(str(scalers_dir)+'/target_scaler_v.save') 

## Load test data

In [ ]:
Np = len(param_list_test)

mesh = dl.load_mesh(data_dir/"cf0025")
snaps = [np.where(mesh['t']/60/60/24==test_days[0])[0][0], np.where(mesh['t']/60/60/24==test_days[1])[0][0]]
Nt_snaps = mesh['t'][snaps[0]:snaps[1]].shape[0]

data_dict = {}
for inx, param in enumerate(param_list_test):
    dirname = 'cf'+str(param).split('.')[1];
    data_dict[inx] = dl.load_variables(data_dir/dirname)

Nn = mesh['nodes'].shape[0]
Ne = mesh['triangles'].shape[0]
Nt = mesh['t'].shape[0]

In [ ]:
dataset_h = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_u = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_v = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed

key = 'h'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_h = np.vstack((dataset_h, snap))

key = 'u'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_u = np.vstack((dataset_u, snap))

key = 'v'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_v = np.vstack((dataset_v, snap))
    
dataset_u_resh = np.reshape(dataset_u,(Np,Nt,Nn))
dataset_u_crop = dataset_u_resh[:,snaps[0]:snaps[1],:]
dataset_u = np.reshape(dataset_u_crop,(Np*Nt_snaps,Nn))

dataset_h_resh = np.reshape(dataset_h,(Np,Nt,Nn))
dataset_h_crop = dataset_h_resh[:,snaps[0]:snaps[1],:]
dataset_h = np.reshape(dataset_h_crop,(Np*Nt_snaps,Nn))

dataset_v_resh = np.reshape(dataset_v,(Np,Nt,Nn))
dataset_v_crop = dataset_v_resh[:,snaps[0]:snaps[1],:]
dataset_v = np.reshape(dataset_v_crop,(Np*Nt_snaps,Nn))

dataset_r_h = np.reshape(dataset_h,(Np,Nt_snaps,Nn))
dataset_r_u = np.reshape(dataset_u,(Np,Nt_snaps,Nn))
dataset_r_v = np.reshape(dataset_v,(Np,Nt_snaps,Nn))


In [ ]:
window_size=5
b_data_h, t_data_h, target_data_h = pu.full_stacked_param_hard_windows(param_list_test,'h',snaps[0],snaps[1],data_dir)
b_data_u, t_data_u, target_data_u = pu.full_stacked_param_hard_windows(param_list_test,'u',snaps[0],snaps[1],data_dir)
b_data_v, t_data_v, target_data_v = pu.full_stacked_param_hard_windows(param_list_test,'v',snaps[0],snaps[1],data_dir)

In [ ]:
if scaling is True:
    t_data_h[:,2] = t_data_h[:,2]/tt_scaler_h
    t_data_h[:,0] = tx_scaler_h.transform(t_data_h[:,0].reshape(-1, 1))[:,0]
    t_data_h[:,1] = ty_scaler_h.transform(t_data_h[:,1].reshape(-1, 1))[:,0]
    b_data_h[:,1:] = b_scaler_h.transform(b_data_h[:,1:])
    target_data_h = target_scaler_h.transform(target_data_h)

    t_data_u[:,2] = t_data_u[:,2]/tt_scaler_u
    t_data_u[:,0] = tx_scaler_u.transform(t_data_u[:,0].reshape(-1, 1))[:,0]
    t_data_u[:,1] = ty_scaler_u.transform(t_data_u[:,1].reshape(-1, 1))[:,0]
    b_data_u[:,1:] = b_scaler_u.transform(b_data_u[:,1:])
    target_data_u = target_scaler_u.transform(target_data_u)

    t_data_v[:,2] = t_data_v[:,2]/tt_scaler_v
    t_data_v[:,0] = tx_scaler_v.transform(t_data_v[:,0].reshape(-1, 1))[:,0]
    t_data_v[:,1] = ty_scaler_v.transform(t_data_v[:,1].reshape(-1, 1))[:,0]
    b_data_v[:,1:] = b_scaler_v.transform(b_data_v[:,1:])
    target_data_v = target_scaler_v.transform(target_data_v)

In [ ]:
b_data_h = np.reshape(b_data_h,(Np,int(Nt_snaps/5),1+Nn+75*window_size))
target_data_h = np.reshape(target_data_h,(Np,int(Nt_snaps/5),Nn*window_size))

b_data_u = np.reshape(b_data_u,(Np,int(Nt_snaps/5),1+Nn+75*window_size))
target_data_u = np.reshape(target_data_u,(Np,int(Nt_snaps/5),Nn*window_size))

b_data_v = np.reshape(b_data_v,(Np,int(Nt_snaps/5),1+Nn+75*window_size))
target_data_v = np.reshape(target_data_v,(Np,int(Nt_snaps/5),Nn*window_size))

### Predictions 1

In [ ]:
res_h = np.zeros((Np, Nt_snaps, Nn))

for p in tqdm(range(Np)):
    b_ic = b_data_h[p, 0, 1:Nn+1]
    res_h[p,0,:] = b_ic
    count = 0
    for i in range(0, int(Nt_snaps/window_size)):
        if (count+window_size) < Nt_snaps:
            b_input = b_data_h[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:Nn+1] = b_ic
            t_input = t_data_h
            
            prediction = mdon_model_h((b_input, t_input)).numpy().reshape(window_size,Nn)
            res_h[p,count+1:count+window_size+1] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
            count = count + window_size
        else:
            diff = Nt_snaps - count - 1
            b_input = b_data_h[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:Nn+1] = b_ic
            t_input = t_data_h
            
            prediction = mdon_model_h((b_input, t_input)).numpy().reshape(window_size,Nn)
            res_h[p,count+1:count+diff+1] = prediction[:diff,:]
                    


full_res_h = target_scaler_h.inverse_transform(res_h.reshape(int(Np*Nt_snaps/window_size),int(Nn*window_size)))

# Reshape data for analysis and visualization
full_res_r_h = np.reshape(full_res_h, (Np, Nt_snaps, Nn))

In [ ]:
res_u = np.zeros((Np, Nt_snaps, Nn))

for p in tqdm(range(Np)):
    b_ic = b_data_u[p, 0, 1:Nn+1]
    res_u[p,0,:] = b_ic
    count = 0
    for i in range(0, int(Nt_snaps/window_size)):
        if (count+window_size) < Nt_snaps:
            b_input = b_data_u[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:Nn+1] = b_ic
            t_input = t_data_u
            
            prediction = mdon_model_u((b_input, t_input)).numpy().reshape(window_size,Nn)
            res_u[p,count+1:count+window_size+1] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
            count = count + window_size
        else:
            diff = Nt_snaps - count - 1
            b_input = b_data_u[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:Nn+1] = b_ic
            t_input = t_data_u
            
            prediction = mdon_model_u((b_input, t_input)).numpy().reshape(window_size,Nn)
            res_u[p,count+1:count+diff+1] = prediction[:diff,:]
                    


full_res_u = target_scaler_u.inverse_transform(res_u.reshape(int(Np*Nt_snaps/window_size),int(Nn*window_size)))

# Reshape data for analysis and visualization
full_res_r_u = np.reshape(full_res_u, (Np, Nt_snaps, Nn))

In [ ]:
res_v = np.zeros((Np, Nt_snaps, Nn))

for p in tqdm(range(Np)):
    b_ic = b_data_v[p, 0, 1:Nn+1]
    res_v[p,0,:] = b_ic
    count = 0
    for i in range(0, int(Nt_snaps/window_size)):
        if (count+window_size) < Nt_snaps:
            b_input = b_data_v[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:Nn+1] = b_ic
            t_input = t_data_v
            
            prediction = mdon_model_v((b_input, t_input)).numpy().reshape(window_size,Nn)
            res_v[p,count+1:count+window_size+1] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
            count = count + window_size
        else:
            diff = Nt_snaps - count - 1
            b_input = b_data_v[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:Nn+1] = b_ic
            t_input = t_data_v
            
            prediction = mdon_model_v((b_input, t_input)).numpy().reshape(window_size,Nn)
            res_v[p,count+1:count+diff+1] = prediction[:diff,:]
                    


full_res_v = target_scaler_v.inverse_transform(res_v.reshape(int(Np*Nt_snaps/window_size),int(Nn*window_size)))

# Reshape data for analysis and visualization
full_res_r_v = np.reshape(full_res_v, (Np, Nt_snaps, Nn))

In [ ]:
np.savez('mdon_output.npz', h_data=full_res_r_h, u_data=full_res_r_u, v_data=full_res_r_v)

In [ ]:
mask_num = 0
rmse_h = np.zeros((Np,Nt_snaps))
rmse_u = np.zeros((Np,Nt_snaps))
rmse_v = np.zeros((Np,Nt_snaps))

max_array_h = np.zeros(Np)
max_array_u = np.zeros(Np)
max_array_v = np.zeros(Np)

n_rmse_h = np.zeros((Np,Nt_snaps))
n_rmse_u = np.zeros((Np,Nt_snaps))
n_rmse_v = np.zeros((Np,Nt_snaps))

for p, param in enumerate(param_list_test[0:]):
    rmse_h[p] = np.sqrt(np.mean((full_res_r_h[p, :, mask_num:] - dataset_r_h[p, :, mask_num:]) ** 2, axis=1))*100
    rmse_u[p] = np.sqrt(np.mean((full_res_r_u[p, :, mask_num:] - dataset_r_u[p, :, mask_num:]) ** 2, axis=1))*100
    rmse_v[p] = np.sqrt(np.mean((full_res_r_v[p, :, mask_num:] - dataset_r_v[p, :, mask_num:]) ** 2, axis=1))*100

    max_array_h[p] = (np.abs(np.max(dataset_r_h[p, :, mask_num:])-np.min(dataset_r_h[p,  :, mask_num:])))*100
    max_array_u[p] = (np.abs(np.max(dataset_r_u[p, :, mask_num:])-np.min(dataset_r_u[p,  :, mask_num:])))*100
    max_array_v[p] = (np.abs(np.max(dataset_r_v[p, :, mask_num:])-np.min(dataset_r_v[p,  :, mask_num:])))*100

    n_rmse_h[p] = rmse_h[p]/max_array_h[p]
    n_rmse_u[p] = rmse_u[p]/max_array_u[p]
    n_rmse_v[p] = rmse_v[p]/max_array_v[p]
    

#### ACC and RMSE Table

In [ ]:
##### ACC
P = dataset_r_h.shape[0]
MM = dataset_r_h.shape[1]
N = dataset_r_h.shape[2]

ACC_h = np.zeros(dataset_r_h.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_h[p,j,:]*100)/N
        Tj = np.sum(dataset_r_h[p,j,:]*100)/N
        num[j] = np.sum((dataset_r_h[p,j,:]*100-Tj)*(full_res_r_h[p,j,:]*100-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_h[p,j,:]*100-Tj)**2)*np.sum((full_res_r_h[p,j,:]*100-Pj)**2))
    ACC_h[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_h.shape[2]):
            
print('ACC:'+str(list(ACC_h)))
print('RMSE:'+str(list(np.mean(rmse_h,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_h,axis=1))))


In [ ]:
##### ACC

P = dataset_r_u.shape[0]
MM = dataset_r_u.shape[1]
N = dataset_r_u.shape[2]

ACC_u = np.zeros(dataset_r_u.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_u[p,j,:]*100)/N
        Tj = np.sum(dataset_r_u[p,j,:]*100)/N
        num[j] = np.sum((dataset_r_u[p,j,:]*100-Tj)*(full_res_r_u[p,j,:]*100-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_u[p,j,:]*100-Tj)**2)*np.sum((full_res_r_u[p,j,:]*100-Pj)**2))
    ACC_u[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_u.shape[2]):
            
print('ACC:'+str(list(ACC_u)))
print('RMSE:'+str(list(np.mean(rmse_u,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_u,axis=1))))


In [ ]:
##### ACC

P = dataset_r_v.shape[0]
MM = dataset_r_v.shape[1]
N = dataset_r_v.shape[2]

ACC_v = np.zeros(dataset_r_v.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_v[p,j,:]*100)/N
        Tj = np.sum(dataset_r_v[p,j,:]*100)/N
        num[j] = np.sum((dataset_r_v[p,j,:]*100-Tj)*(full_res_r_v[p,j,:]*100-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_v[p,j,:]*100-Tj)**2)*np.sum((full_res_r_v[p,j,:]*100-Pj)**2))
    ACC_v[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_v.shape[2]):
            
print('ACC:'+str(list(ACC_v)))
print('RMSE:'+str(list(np.mean(rmse_v,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_v,axis=1))))


### Predictions 2

In [ ]:
# day_window = 5.
# day_total = test_days[1]-test_days[0]

# n_windows = int(day_total/day_window)

# snap_windows = int(Nt_snaps/day_total*day_window)

# rmse_array_h = np.zeros((n_windows, len(param_list_test)))
# rmse_array_u = np.zeros((n_windows, len(param_list_test)))
# rmse_array_v = np.zeros((n_windows, len(param_list_test)))

# max_array_h = np.zeros((n_windows, len(param_list_test)))
# max_array_u = np.zeros((n_windows, len(param_list_test)))
# max_array_v = np.zeros((n_windows, len(param_list_test)))

In [ ]:
# current_window = 0
# current_window2 = 0

# for w in tqdm(range(n_windows)):
#     res_h = np.zeros((Np, snap_windows, Nn))
#     res_u = np.zeros((Np, snap_windows, Nn))
#     res_v = np.zeros((Np, snap_windows, Nn))
    
#     for p in range(Np):

#         b_ic_h = b_data_h[p, current_window, 1:Nn+1]
#         res_h[p,0,:] = b_ic_h
        
#         b_ic_u = b_data_u[p, current_window, 1:Nn+1]
#         res_u[p,0,:] = b_ic_u
        
#         b_ic_v = b_data_v[p, current_window, 1:Nn+1]
#         res_v[p,0,:] = b_ic_v
        
#         count = 0
#         for i in range(0, int(snap_windows/window_size)):
#             temp_i = i + current_window

#             if (count+window_size+1) < snap_windows:
                
#                 b_input_h = b_data_h[p, temp_i, :].reshape(1,-1).copy()
#                 b_input_h[0, 1:Nn+1] = b_ic_h
#                 t_input_h = t_data_h
#                 prediction_h = mdon_model_h((b_input_h, t_input_h)).numpy().reshape(window_size,Nn)
#                 res_h[p,count+1:count+window_size+1] = prediction_h
#                 b_ic_h = prediction_h[-1]  

#                 b_input_u = b_data_u[p, temp_i, :].reshape(1,-1).copy()
#                 b_input_u[0, 1:Nn+1] = b_ic_u
#                 t_input_u = t_data_u
#                 prediction_u = mdon_model_u((b_input_u, t_input_u)).numpy().reshape(window_size,Nn)
#                 res_u[p,count+1:count+window_size+1] = prediction_u
#                 b_ic_u = prediction_u[-1]  
                
#                 b_input_v = b_data_v[p, temp_i, :].reshape(1,-1).copy()
#                 b_input_v[0, 1:Nn+1] = b_ic_v
#                 t_input_v = t_data_v
#                 prediction_v = mdon_model_v((b_input_v, t_input_v)).numpy().reshape(window_size,Nn)
#                 res_v[p,count+1:count+window_size+1] = prediction_v
#                 b_ic_v = prediction_v[-1]  
                
#                 count = count + window_size
               
#             else:
#                 diff = snap_windows - count - 1
                
#                 b_input_h = b_data_h[p, temp_i, :].reshape(1,-1).copy()
#                 b_input_h[0, 1:Nn+1] = b_ic_h
#                 t_input_h = t_data_h
#                 prediction_h = mdon_model_h((b_input_h, t_input_h)).numpy().reshape(window_size,Nn)
#                 res_h[p,count+1:count+diff+1] = prediction_h[0:diff,:]
#                 b_ic_h = prediction_h[-1] 

#                 b_input_u = b_data_u[p, temp_i, :].reshape(1,-1).copy()
#                 b_input_u[0, 1:Nn+1] = b_ic_u
#                 t_input_u = t_data_u
#                 prediction_u = mdon_model_u((b_input_u, t_input_u)).numpy().reshape(window_size,Nn)
#                 res_u[p,count+1:count+diff+1] = prediction_u[0:diff,:]
#                 b_ic_u = prediction_u[-1] 
                
#                 b_input_v = b_data_v[p, temp_i, :].reshape(1,-1).copy()
#                 b_input_v[0, 1:Nn+1] = b_ic_v
#                 t_input_v = t_data_v
#                 prediction_v = mdon_model_v((b_input_v, t_input_v)).numpy().reshape(window_size,Nn)
#                 res_v[p,count+1:count+diff+1] = prediction_v[0:diff,:]
#                 b_ic_v = prediction_v[-1] 
                

#     res_h = target_scaler_h.inverse_transform(res_h.reshape(int(Np*snap_windows/window_size),int(Nn*window_size)))
#     full_res_r_h = np.reshape(res_h, (Np, snap_windows, Nn))

#     res_u = target_scaler_u.inverse_transform(res_u.reshape(int(Np*snap_windows/window_size),int(Nn*window_size)))
#     full_res_r_u = np.reshape(res_u, (Np, snap_windows, Nn))

#     res_v = target_scaler_v.inverse_transform(res_v.reshape(int(Np*snap_windows/window_size),int(Nn*window_size)))
#     full_res_r_v = np.reshape(res_v, (Np, snap_windows, Nn))

#     for p in range(Np):

#         rmse_array_h[w, p] = np.mean(np.sqrt(np.mean((full_res_r_h[p, :] - dataset_r_h[p, current_window2:snap_windows+current_window2] ) ** 2, axis=1)))
#         max_array_h[w, p] = (np.abs(np.max(dataset_r_h[p, current_window2:snap_windows+current_window2])-np.min(dataset_r_h[p, current_window2:snap_windows+current_window2])))
        
#         rmse_array_u[w, p] = np.mean(np.sqrt(np.mean((full_res_r_u[p, :] - dataset_r_u[p, current_window2:snap_windows+current_window2] ) ** 2, axis=1)))
#         max_array_u[w, p] = (np.abs(np.max(dataset_r_u[p, current_window2:snap_windows+current_window2])-np.min(dataset_r_u[p, current_window2:snap_windows+current_window2])))
        
#         rmse_array_v[w, p] = np.mean(np.sqrt(np.mean((full_res_r_v[p, :] - dataset_r_v[p, current_window2:snap_windows+current_window2] ) ** 2, axis=1)))
#         max_array_v[w, p] = (np.abs(np.max(dataset_r_v[p, current_window2:snap_windows+current_window2])-np.min(dataset_r_v[p, current_window2:snap_windows+current_window2])))

#     current_window += int(snap_windows/window_size)
#     current_window2 += snap_windows


### Predictions 3

In [ ]:
# if scaling:
#     zero_ic_h = target_scaler_h.transform(np.zeros((1,5*Nn)))[:,0:Nn]
#     zero_ic_u = target_scaler_u.transform(np.zeros((1,5*Nn)))[:,0:Nn]
#     zero_ic_v = target_scaler_v.transform(np.zeros((1,5*Nn)))[:,0:Nn]

In [ ]:
# snaps_crop = [np.where(mesh['t']/60/60/24==45.)[0][0], np.where(mesh['t']/60/60/24==55.)[0][0]]
# Nt_snaps_crop = mesh['t'][snaps_crop[0]:snaps_crop[1]].shape[0]
# snaps_crop -= snaps[0]


In [ ]:
# res_h = np.zeros((Np, Nt_snaps_crop, Nn))

# for p in tqdm(range(Np)):
#     b_ic = zero_ic_h[0]
#     res_h[p,0,:] = b_ic
#     count = 0
#     for i in range(int(snaps_crop[0]/window_size),int(snaps_crop[1]/window_size)):
#         if (count+window_size) < Nt_snaps_crop:
#             b_input = b_data_h[p, i, :].reshape(1,-1).copy()
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_h
            
#             prediction = mdon_model_h((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_h[p,count+1:count+window_size+1] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#             count = count + window_size
#         else:
#             diff = Nt_snaps_crop - count - 1
#             b_input = b_data_h[p, i, :].reshape(1,-1).copy()
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_h
            
#             prediction = mdon_model_h((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_h[p,count+1:count+diff+1] = prediction[:diff,:]
                    


# full_res_h = target_scaler_h.inverse_transform(res_h.reshape(int(Np*Nt_snaps_crop/window_size),int(Nn*window_size)))

# # Reshape data for analysis and visualization
# full_res_r_h = np.reshape(full_res_h, (Np, Nt_snaps_crop, Nn))

In [ ]:
# # res_u = np.zeros((Np, Nt_snaps_crop, Nn))

# # for p in tqdm(range(Np)):
# #     b_ic = zero_ic_u[0]
# #     res_u[p,0,:] = b_ic
# #     count = 0
# #     for i in range(int(snaps_crop[0]/window_size),int(snaps_crop[1]/window_size)):
# #         if (count+window_size) < Nt_snaps_crop:
# #             b_input = b_data_u[p, i, :].reshape(1,-1).copy()
# #             b_input[0, 1:Nn+1] = b_ic
# #             t_input = t_data_u
            
# #             prediction = mdon_model_u((b_input, t_input)).numpy().reshape(window_size,Nn)
# #             res_u[p,count+1:count+window_size+1] = prediction
# #             b_ic = prediction[-1]  # Update initial condition for the next iteration
# #             count = count + window_size
# #         else:
# #             diff = Nt_snaps_crop - count - 1
# #             b_input = b_data_u[p, i, :].reshape(1,-1).copy()
# #             b_input[0, 1:Nn+1] = b_ic
# #             t_input = t_data_u
            
# #             prediction = mdon_model_u((b_input, t_input)).numpy().reshape(window_size,Nn)
# #             res_u[p,count+1:count+diff+1] = prediction[:diff,:]
                    


# # full_res_u = target_scaler_u.inverse_transform(res_u.reshape(int(Np*Nt_snaps_crop/window_size),int(Nn*window_size)))

# # Reshape data for analysis and visualization
# full_res_r_u = np.reshape(full_res_u, (Np, Nt_snaps_crop, Nn))

In [ ]:
# res_v = np.zeros((Np, Nt_snaps_crop, Nn))

# for p in tqdm(range(Np)):
#     b_ic = zero_ic_v[0]
#     res_v[p,0,:] = b_ic
#     count = 0
#     for i in range(int(snaps_crop[0]/window_size),int(snaps_crop[1]/window_size)):
#         if (count+window_size) < Nt_snaps_crop:
#             b_input = b_data_v[p, i, :].reshape(1,-1).copy()
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_v
            
#             prediction = mdon_model_v((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_v[p,count+1:count+window_size+1] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#             count = count + window_size
#         else:
#             diff = Nt_snaps_crop - count - 1
#             b_input = b_data_v[p, i, :].reshape(1,-1).copy()
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_v
            
#             prediction = mdon_model_v((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_v[p,count+1:count+diff+1] = prediction[:diff,:]
                    


# full_res_v = target_scaler_v.inverse_transform(res_v.reshape(int(Np*Nt_snaps_crop/window_size),int(Nn*window_size)))

# # Reshape data for analysis and visualization
# full_res_r_v = np.reshape(full_res_v, (Np, Nt_snaps_crop, Nn))

### Predictions 4

In [ ]:
# res_h = np.zeros((Np, Nt_snaps_crop, Nn))

# for p in tqdm(range(Np)):
#     b_ic = b_data_h[p,int((snaps_crop/window_size)[0]),1:Nn+1]
#     res_h[p,0,:] = b_ic
#     count = 0
#     for i in range(int(snaps_crop[0]/window_size),int(snaps_crop[1]/window_size)):
#         if (count+window_size) < Nt_snaps_crop:
#             b_input = b_data_h[p, i, :].reshape(1,-1)
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_h
            
#             prediction = mdon_model_h((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_h[p,count+1:count+window_size+1] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#             count = count + window_size
#         else:
#             diff = Nt_snaps_crop - count - 1
#             b_input = b_data_h[p, i, :].reshape(1,-1)
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_h
            
#             prediction = mdon_model_h((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_h[p,count+1:count+diff+1] = prediction[:diff,:]
                    

# full_res_h = target_scaler_h.inverse_transform(res_h.reshape(int(Np*Nt_snaps_crop/window_size),int(Nn*window_size)))

# # Reshape data for analysis and visualization
# full_res_r_h = np.reshape(full_res_h, (Np, Nt_snaps_crop, Nn))

In [ ]:
# res_u = np.zeros((Np, Nt_snaps_crop, Nn))

# for p in tqdm(range(Np)):
#     b_ic = b_data_u[p,int((snaps_crop/window_size)[0]),1:Nn+1]    
#     res_u[p,0,:] = b_ic
#     count = 0
#     for i in range(int(snaps_crop[0]/window_size),int(snaps_crop[1]/window_size)):
#         if (count+window_size) < Nt_snaps_crop:
#             b_input = b_data_u[p, i, :].reshape(1,-1).copy()
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_u
            
#             prediction = mdon_model_u((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_u[p,count+1:count+window_size+1] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#             count = count + window_size
#         else:
#             diff = Nt_snaps_crop - count - 1
#             b_input = b_data_u[p, i, :].reshape(1,-1).copy()
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_u
            
#             prediction = mdon_model_u((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_u[p,count+1:count+diff+1] = prediction[:diff,:]
                    


# full_res_u = target_scaler_u.inverse_transform(res_u.reshape(int(Np*Nt_snaps_crop/window_size),int(Nn*window_size)))

# # Reshape data for analysis and visualization
# full_res_r_u = np.reshape(full_res_u, (Np, Nt_snaps_crop, Nn))

In [ ]:
# res_v = np.zeros((Np, Nt_snaps_crop, Nn))

# for p in tqdm(range(Np)):
#     b_ic = b_data_v[p,int((snaps_crop/window_size)[0]),1:Nn+1]
#     res_v[p,0,:] = b_ic
#     count = 0
#     for i in range(int(snaps_crop[0]/window_size),int(snaps_crop[1]/window_size)):
#         if (count+window_size) < Nt_snaps_crop:
#             b_input = b_data_v[p, i, :].reshape(1,-1).copy()
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_v
            
#             prediction = mdon_model_v((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_v[p,count+1:count+window_size+1] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#             count = count + window_size
#         else:
#             diff = Nt_snaps_crop - count - 1
#             b_input = b_data_v[p, i, :].reshape(1,-1).copy()
#             b_input[0, 1:Nn+1] = b_ic
#             t_input = t_data_v
            
#             prediction = mdon_model_v((b_input, t_input)).numpy().reshape(window_size,Nn)
#             res_v[p,count+1:count+diff+1] = prediction[:diff,:]
                    


# full_res_v = target_scaler_v.inverse_transform(res_v.reshape(int(Np*Nt_snaps_crop/window_size),int(Nn*window_size)))

# # Reshape data for analysis and visualization
# full_res_r_v = np.reshape(full_res_v, (Np, Nt_snaps_crop, Nn))